# **Latin Grammar Question Intent Classifier**

Fine-tuning a BERT-based model for intent classification to identify user grammatical queries.

## **Import Dependencies**

In [1]:
import os
os.environ["WANDB_DISABLED"] = "true"

In [2]:
import transformers
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments
from datasets import load_dataset, Dataset
import pandas as pd
import numpy as np
import torch
import json
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, recall_score
import matplotlib.pyplot as plt

In [3]:
%matplotlib inline

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
os.chdir('/content/drive/MyDrive/FYP')

In [6]:
from huggingface_hub import login
from google.colab import userdata

login(token=userdata.get('HF_TOKEN'))

In [7]:
print(torch.cuda.is_available())
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

True
Using device: cuda


## **Prepare Data**

### Load Dataset

In [8]:
DATA_PATH = os.path.join("data_test.csv")

In [9]:
# load necessary columns into df
df = pd.read_csv(DATA_PATH, usecols=['question', 'intent'])
df.head()

,intent,question
0,part_of_speech,What is the POS of algēnsis?
1,part_of_speech,What part of speech is algēnse?
2,part_of_speech,What is the part of speech of algēnsēs?
3,part_of_speech,Does algēnsia function as a noun?
4,part_of_speech,What POS is algēnsis?


In [10]:
# dataset summary
print(f"Dataset size: {len(df)} rows")
print(f"Intents: {df['intent'].value_counts().to_dict()}")

Dataset size: 8113 rows
Intents: {'part_of_speech': 3500, 'tense': 1656, 'voice': 1656, 'person': 482, 'mood': 482, 'aspect': 337}


### Split into Train/Test/Validation Sets

In [11]:
# 70% train, 15% test, 15% val
train_df, test_val_df = train_test_split(df, test_size=0.3, random_state=42)
test_df, val_df = train_test_split(test_val_df, test_size=0.5, random_state=42)

In [12]:
print(len(train_df), "train samples")
print(len(test_df), "test samples")
print(len(val_df), "validation samples")

5679 train samples
1217 test samples
1217 validation samples


### Data Preprocessing

In [13]:
# encode intents as numerical labels
intents = sorted(train_df['intent'].unique())
print("Total number of intents:", len(intents))

intent2label = {intent: i for i, intent in enumerate(intents)} # label mappings
label2intent = {i: intent for i, intent in enumerate(intents)}

print("Intent to Label:", intent2label)
print("Label to Intent:", label2intent)

Total number of intents: 6
Intent to Label: {'aspect': 0, 'mood': 1, 'part_of_speech': 2, 'person': 3, 'tense': 4, 'voice': 5}
Label to Intent: {0: 'aspect', 1: 'mood', 2: 'part_of_speech', 3: 'person', 4: 'tense', 5: 'voice'}


In [14]:
# map intents in df to numerical label
train_df['label'] = train_df['intent'].map(intent2label)
val_df['label']   = val_df['intent'].map(intent2label)
test_df['label']  = test_df['intent'].map(intent2label)

train_df.head()

,intent,question,label
2402,part_of_speech,What is the POS of repedātum?,2
5802,tense,"What is the tense of adsellātūrus: present, pa...",4
1073,part_of_speech,What part of speech is hippotoxotae?,2
5584,tense,Is frūnīscundus past tense?,4
144,part_of_speech,Is inclēmentem a participle?,2


### Tokenize Dataset

In [15]:
# load tokenizer
# tokenizer = BertTokenizer.from_pretrained("bert-large-uncased")
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [16]:
# tokenize function
def tokenize(batch):
    return tokenizer(
        batch["question"],
        truncation=True,
        padding="max_length",
        max_length=50
    )

In [17]:
# convert dataframes to type Dataset
train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)
test_dataset = Dataset.from_pandas(test_df)

In [18]:
# tokenize the dataset
train_dataset = train_dataset.map(tokenize, batched=True)
val_dataset = val_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)

Map:   0%|          | 0/5679 [00:00<?, ? examples/s]

Map:   0%|          | 0/1217 [00:00<?, ? examples/s]

Map:   0%|          | 0/1217 [00:00<?, ? examples/s]

In [19]:
print(train_dataset[0])

{'intent': 'part_of_speech', 'question': 'What is the POS of repedātum?', 'label': 2, '__index_level_0__': 2402, 'input_ids': [101, 2054, 2003, 1996, 13433, 2015, 1997, 16360, 11960, 11667, 1029, 102, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]}


### Data Visualisation

## **Model**

### Load Model

In [20]:
# model = BertForSequenceClassification.from_pretrained('bert-large-uncased', num_labels=len(intents), id2label=label2intent, label2id=intent2label)
model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=len(intents), id2label=label2intent, label2id=intent2label)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [21]:
# define training args
args = TrainingArguments(
    output_dir="./bert_finetuned",          # Directory to save model checkpoints
    eval_strategy="epoch",                  # Evaluate on validation set every epoch
    save_strategy="epoch",                  # Save model checkpoints every epoch
    learning_rate=4e-5,                     # Learning rate
    warmup_steps=0.1,                       # Proportion of training steps to linearly increase learning rate
    per_device_train_batch_size=32,         # Batch size for training
    gradient_accumulation_steps=1,
    num_train_epochs=10,                    # Number of training epochs
    weight_decay=0.01,                      # Regularization to prevent overfitting
    load_best_model_at_end=True,            # Load the best model based on evaluation metrics
    save_total_limit=1,                     # Maximum number of checkpoints to keep - keeps best and most recent
    fp16=True                               # Enable mixed precision for faster training
)

### Training

In [22]:
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    # compute_metrics=compute_metrics
)

In [23]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,No log,0.005066
2,No log,0.001215
3,0.263748,0.000688
4,0.263748,0.000479
5,0.263748,0.000372
6,0.000697,0.000308
7,0.000697,0.000269
8,0.000697,0.000244
9,0.000392,0.000230
10,0.000392,0.000225


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=1780, training_loss=0.07444488866191902, metrics={'train_runtime': 395.3888, 'train_samples_per_second': 143.631, 'train_steps_per_second': 4.502, 'total_flos': 1459239596622000.0, 'train_loss': 0.07444488866191902, 'epoch': 10.0})

## **Evaluation**